In [ ]:
# ==========================================
# TESIS ZMVM: PM, clima y salud
# Autor: Arely Leal
# Descripción: genera heatmaps de la FAP para material particulado 
# Periodo: PM10 (2000-2019) y PM2.5 (2003-2019)
# ==========================================

In [17]:

import pandas as pd
import os
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from matplotlib.colors import LinearSegmentedColormap

# =========================
# CONFIGURACIÓN DE RUTAS
# =========================
archivo_entrada = "RUTA DEL ARCHIVO"
carpeta = os.path.dirname(archivo_entrada)
carpeta_salida = os.path.join(carpeta, "NOMBRE DE SALIDA")
os.makedirs(carpeta_salida, exist_ok=True)

# =========================
# PROCESAMIENTO DE DATOS
# =========================
df = pd.read_csv(archivo_entrada)

df["Anio"] = pd.to_numeric(df["Anio"], errors="coerce")
df["FAP_%"] = pd.to_numeric(df["FAP_%"], errors="coerce")
df["hora"] = pd.to_numeric(df["hora"], errors="coerce")

edad_order = ["00_04", "05_14", "15_24", "25_34", "35_44", "45_54", "55_64", "65+"]
edad_labels = ["0-4", "5-14", "15-24", "25-34", "35-44", "45-54", "55-64", "65+"]

cat_order = ["M", "MM", "EM"]
cat_nombres = {"M": "Mala", "MM": "Muy Mala", "EM": "Extremadamente Mala"}

horas_objetivo = [8, 18]
hora_nombres = {8: "8:00 horas", 18: "18:00 horas"}

# =========================
# COLORES NORMATIVOS
# =========================
def crear_cmap(rgb_color):
    cmap = LinearSegmentedColormap.from_list(
        "custom_cmap", ["#ffffff", rgb_color], N=256
    )
    cmap.set_bad("#f6f6f6")  # gris muy claro para valores sin datos
    return cmap

colores_categorias = {
    "M": crear_cmap((255/255, 126/255, 0/255)),
    "MM": crear_cmap((255/255, 0/255, 0/255)),
    "EM": crear_cmap((143/255, 63/255, 151/255))
}

global_max = max(float(df["FAP_%"].max()), 1)
sns.set_style("white")

# =========================
# GENERACIÓN DE FIGURAS (2x3) POR HORA
# =========================
for enfermedad in df["Enfermedad"].dropna().unique():
    df_enf = df[df["Enfermedad"] == enfermedad].copy()

    for hora in horas_objetivo:
        df_enf_hora = df_enf[df_enf["hora"] == hora].copy()

        if df_enf_hora.empty:
            print(f"Sin datos para {enfermedad} a las {hora_nombres[hora]}")
            continue

        fig = plt.figure(figsize=(26, 13))
        outer = fig.add_gridspec(2, 3, wspace=0.20, hspace=0.30)

        for row, sexo_label in enumerate(["MUJERES", "HOMBRES"]):
            for col, cat in enumerate(cat_order):

                inner = outer[row, col].subgridspec(
                    1, 2, width_ratios=[1, 0.05], wspace=0.10
                )
                ax = fig.add_subplot(inner[0, 0])
                cax = fig.add_subplot(inner[0, 1])

                sub_data = df_enf_hora[
                    (df_enf_hora["Categoria_PM"] == cat) &
                    (df_enf_hora["Sexo"] == sexo_label)
                ].copy()

                if sub_data.empty:
                    ax.axis("off")
                    cax.axis("off")
                    continue

                tabla = sub_data.pivot_table(
                    index="Edad_gpo",
                    columns="Anio",
                    values="FAP_%",
                    aggfunc="mean"
                ).reindex(index=edad_order)

                tabla = tabla.reindex(sorted(tabla.columns), axis=1)

                sns.heatmap(
                    tabla,
                    ax=ax,
                    cmap=colores_categorias[cat],
                    vmin=0,
                    vmax=global_max,
                    linewidths=0.5,
                    linecolor="#f9f9f9",
                    cbar=True,
                    cbar_ax=cax,
                    cbar_kws={"label": r"$\bf{FAP\ (\%)}$"}
                )

                cax.yaxis.labelpad = 10

                if row == 0:
                    ax.set_title(
                        f"{cat_nombres[cat]}",
                        fontsize=16,
                        pad=25,
                        fontweight="bold"
                    )

                if col == 0:
                    ax.annotate(
                        sexo_label.capitalize(),
                        xy=(-0.16, 0.5),
                        xycoords="axes fraction",
                        ha="center",
                        va="center",
                        fontsize=18,
                        fontweight="normal",
                        rotation=90
                    )

                ax.set_xlabel("Año", fontweight="bold", fontsize=12)
                ax.set_ylabel("Grupo de edad", fontweight="bold", fontsize=12)
                ax.set_yticklabels(edad_labels, rotation=0, fontsize=11)

                years = tabla.columns.tolist()
                ax.set_xticks(np.arange(len(years)) + 0.5)
                ax.set_xticklabels(
                    [str(int(y)) if idx % 2 == 0 else "" for idx, y in enumerate(years)],
                    rotation=45,
                    ha="right",
                    fontsize=10
                )

        fig.text(
            0.5, 0.96,
            "FRACCIÓN ATRIBUIBLE POBLACIONAL ($PM_{MODIFICAR 10/2.5}$)",
            ha="center",
            fontsize=24,
            fontweight="bold"
        )
        fig.text(
            0.5, 0.93,
            f"{enfermedad.title()} - {hora_nombres[hora]}",
            ha="center",
            fontsize=20,
            fontweight="normal"
        )

        plt.subplots_adjust(top=0.86, bottom=0.10, left=0.07, right=0.95)

        nombre_limpio = enfermedad.replace(" ", "_").replace("/", "-")
        nombre_salida = f"PM_2x3_{nombre_limpio}_{hora_nombres[hora]}.png"

        plt.savefig(
            os.path.join(carpeta_salida, nombre_salida),
            dpi=300,
            bbox_inches="tight"
        )
        plt.close()

print("Gráficos generados por hora.")

Gráficos generados por hora.
